In [7]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field 
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from tavily import TavilyClient


In [8]:
# We will create a Travel partner chain , in which model will search the internet and give the output in real time . 

load_dotenv()
# initializing the model 
model = ChatOpenAI(model='gpt-4o-mini')




In [9]:
# Creating a prompt 
prompt = PromptTemplate(
    template = ''' You are an Travel planner , and you will give user the best places  to visit for {days} days in the  {location}. with the real time internet''', 
    input_variables =['days', 'location']
)


days = input('Tell me How many days you want to visit the location')
location = input('Tell me which Location you want to visit ')
# Crafting the  prompt 

In [4]:
@tool
def web_search(location:str, days:int,  max_results:int=5 ) -> str:
    ''' you are web_search tool, which will fetch real time results from the internet and provide output to the user

    location : it will give you the base location in which user has to visit
    days : number of days it has been planned  '''

    full_query = f"Tell me the best places to visit in {location} for {days} days "

    client = TavilyClient()
    response = client.search(query = full_query, search_depth='basic', max_results=max_results)
    real_response = response['results']
    listi =[]
    for i, dicti in enumerate (real_response):
        listi.append(dicti['content'])
    return (listi)






In [10]:
# Now connecting the tool with LLMs . 

model_with_tools = model.bind_tools([web_search])

chain = prompt|model_with_tools
response = chain.invoke({'days':days,'location':location})
print (response)

print (response.tool_calls)

# Our model is working with tools and working fine 

if response.tool_calls :
    tool_call= response.tool_calls[0]
    args = tool_call['args']
    
    output = web_search.invoke(args)
    print (output)





content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 136, 'total_tokens': 157, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_83c2412508', 'id': 'chatcmpl-DqLwsFAckYaGrqbuCQh0owUzV1b0Q', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019ec1ea-0364-7953-a289-1806137dfa55-0' tool_calls=[{'name': 'web_search', 'args': {'location': 'Uttarakhand', 'days': 3}, 'id': 'call_SBGi4g48HpcL8c9ut4Tztl3S', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 136, 'output_tokens': 21, 'total_tokens': 157, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
[{'name'

In [11]:
# Creating a structured Output 

class StructuredOutputSchema(BaseModel):
    location : str = Field (description= 'what is the location user requires')
    places :str = Field(description='The place user will be displaying to the user')
    planning : str = Field (description='understanding the places , fit these into a  days schedule with each day one by one')
    days :str = Field (description= 'Tell me the number of days that person will go for trip ')

structured_model = model.with_structured_output(StructuredOutputSchema)

# We will fetch the structured output from this model 
structured_data = structured_model.invoke(output)
#print (structured_data)

# Now we will create a chain for the final output , and provide the user the output 



prompt = PromptTemplate(
    template = '''You are a an travelling AI assistant , Summarize these {itienary_data}into the following {days} days in 3-4  bullet points
    like - 
    First , give the person an introductory message , and then , we will design your travel itienary for {location}
    Day 1 - 'Shanivar wada ' ,with three bullet points from {itienary_data}

    ''',
    input_variables=['days','itienary_data','location']
)

summary_chain = prompt|model 

for summary_chunk in summary_chain.stream({'days':structured_data.days, 'location':structured_data.location,'itienary_data':structured_data.places}):
    print (summary_chunk.content,end ='')



**Welcome to your Uttarakhand Adventure!** Get ready to explore the breathtaking beauty and spiritual essence of this enchanting region. We’ve designed a 6-day itinerary that captures the highlights of your trip, ensuring an unforgettable experience.

### Day 1 - Kedarnath
- Begin your journey with a visit to **Kedarnath**, one of the sacred Char Dham sites, where you can explore the ancient temple dedicated to Lord Shiva.
- Immerse yourself in the spiritual atmosphere and witness the serene landscapes surrounding the temple, with majestic views of the Himalayas.
- Spend your evening at **Triveni Ghat** in Rishikesh for a tranquil riverside experience.

### Day 2 - Ganga Aarti at Haridwar & Mussoorie
- Attend the mesmerizing **Ganga Aarti** at Haridwar, where the evening rituals take place on the banks of the holy Ganges.
- Afterward, head to **Mussoorie**, known as the "Queen of Hills," and check-in at a cozy spot to enjoy the mountain ambiance.
- Explore the **Lal Tibba** viewpoint f